# CUDA Concept and Programming
## Group 3:

Members:
*   Alcasid, Aizy
*   Bilanes, Kendrick
*   Corpuz, Gerald
*   Roa, Luis







CUDA Version 5: With Prefetch, WIth Page Creation, With Mem Advise

In [ ]:
%%writefile CUDA_axpy5.cu
// CUDA version 5.0 - AXPY: Z[i] = a * A[i] + B[i]
// Grid-stride loop + prefetch + page creation + mem advise
#include <stdio.h>
#include <stdlib.h>
#include <math.h>

#define CHECK(call) do { cudaError_t e = (call); if (e != cudaSuccess) { \
    printf("CUDA error %s at line %d\n", cudaGetErrorString(e), __LINE__); exit(1);} } while(0)

// CUDA AXPY kernel (grid-stride loop)
__global__
void axpy(size_t n, float a, const float *A, const float *B, float *Z){
    size_t index  = (size_t)blockIdx.x * blockDim.x + threadIdx.x;
    size_t stride = (size_t)blockDim.x * gridDim.x;
    for (size_t i = index; i < n; i += stride)
        Z[i] = a * A[i] + B[i];
}

int main(int argc, char **argv){
    const size_t ARRAY_SIZE  = 1 << 24;
    const size_t ARRAY_BYTES = ARRAY_SIZE * sizeof(float);
    const int    loope       = 30;
    const float  a           = 2.0f;

    // Parse block dimensions from command line
    size_t numThreads = (argc > 1) ? (size_t)atoi(argv[1]) : 1024;
    size_t numBlocks  = (ARRAY_SIZE + numThreads - 1) / numThreads; // round-up
    size_t maxBlocks  = (argc > 2) ? (size_t)atoi(argv[2]) : 0;
    if (maxBlocks > 0 && numBlocks > maxBlocks) numBlocks = maxBlocks;

    int device = -1;
    CHECK(cudaGetDevice(&device));

    cudaMemLocation cpuLoc;
    cpuLoc.type = cudaMemLocationTypeHost;
    cpuLoc.id   = 0;

    cudaMemLocation gpuLoc;
    gpuLoc.type = cudaMemLocationTypeDevice;
    gpuLoc.id   = device;

    // 1. Allocate Unified Memory
    float *A, *B, *Z;
    CHECK(cudaMallocManaged(&A, ARRAY_BYTES));
    CHECK(cudaMallocManaged(&B, ARRAY_BYTES));
    CHECK(cudaMallocManaged(&Z, ARRAY_BYTES));

    // 2. Memory Advise
    // Advise the driver that A and B are strictly read-only for the GPU
    CHECK(cudaMemAdvise(A, ARRAY_BYTES, cudaMemAdviseSetReadMostly, gpuLoc));
    CHECK(cudaMemAdvise(B, ARRAY_BYTES, cudaMemAdviseSetReadMostly, gpuLoc));
    // Advise that Z is written by the GPU, making device memory its preferred location
    CHECK(cudaMemAdvise(Z, ARRAY_BYTES, cudaMemAdviseSetPreferredLocation, gpuLoc));

    // 3. Page Creation Setup (Prefetch to Host before initializing)
    CHECK(cudaMemPrefetchAsync(A, ARRAY_BYTES, cpuLoc, 0));
    CHECK(cudaMemPrefetchAsync(B, ARRAY_BYTES, cpuLoc, 0));
    CHECK(cudaMemPrefetchAsync(Z, ARRAY_BYTES, cpuLoc, 0));
    CHECK(cudaDeviceSynchronize());

    // 4. Initialize Data (Creates pages on the Host)
    for (size_t i = 0; i < ARRAY_SIZE; i++){
        A[i] = (float)(i % 1000) * 0.5f;
        B[i] = (float)(i % 777)  * 0.25f;
    }

    // 5. Prefetch to GPU prior to kernel execution
    CHECK(cudaMemPrefetchAsync(A, ARRAY_BYTES, gpuLoc, 0));
    CHECK(cudaMemPrefetchAsync(B, ARRAY_BYTES, gpuLoc, 0));
    CHECK(cudaMemPrefetchAsync(Z, ARRAY_BYTES, gpuLoc, 0));
    CHECK(cudaDeviceSynchronize());

    printf(" *** function = Float AXPY (CUDA Prefetch + Page Creation + Mem Advise)\n");
    printf("numElements = %lu\n", ARRAY_SIZE);
    printf("numBlocks = %lu, numThreads = %lu\n", numBlocks, numThreads);

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    // Warm-up run
    axpy<<<numBlocks, numThreads>>>(ARRAY_SIZE, a, A, B, Z);
    CHECK(cudaDeviceSynchronize());

    // Execution block
    double total = 0.0, tmin = 1e30, tmax = 0.0;
    for (int i = 0; i < loope; i++){
        cudaEventRecord(start);
        axpy<<<numBlocks, numThreads>>>(ARRAY_SIZE, a, A, B, Z);
        cudaEventRecord(stop);
        CHECK(cudaEventSynchronize(stop));

        float ms = 0.0f;
        cudaEventElapsedTime(&ms, start, stop);
        total += ms;
        if (ms < tmin) tmin = ms;
        if (ms > tmax) tmax = ms;
    }
    CHECK(cudaGetLastError());
    printf("Kernel time over %d runs: avg = %f ms, min = %f ms, max = %f ms\n",
           loope, total / loope, tmin, tmax);

    // Prefetch back to CPU for verification
    CHECK(cudaMemPrefetchAsync(Z, ARRAY_BYTES, cpuLoc, 0));
    CHECK(cudaMemPrefetchAsync(A, ARRAY_BYTES, cpuLoc, 0));
    CHECK(cudaMemPrefetchAsync(B, ARRAY_BYTES, cpuLoc, 0));
    CHECK(cudaDeviceSynchronize());

    // Correctness check
    size_t err_count = 0;
    for (size_t i = 0; i < ARRAY_SIZE; i++){
        float ref = a * A[i] + B[i];
        if (fabsf(ref - Z[i]) > 1e-5f * fabsf(ref) + 1e-6f)
            err_count++;
    }
    printf("Error count(CUDA program): %lu\n", err_count);

    // Display first 5 and last 5 elements
    printf("First 5 of Z: ");
    for (int i = 0; i < 5; i++) printf("%f ", Z[i]);
    printf("\nLast 5 of Z:  ");
    for (size_t i = ARRAY_SIZE - 5; i < ARRAY_SIZE; i++) printf("%f ", Z[i]);
    printf("\n\n");

    cudaEventDestroy(start);
    cudaEventDestroy(stop);
    cudaFree(A);
    cudaFree(B);
    cudaFree(Z);

    return 0;
}

Overwriting CUDA_axpy5.cu


In [ ]:
%%shell
nvcc -O2 -arch=native CUDA_axpy5.cu -o CUDA_axpy5
./CUDA_axpy5 256
./CUDA_axpy5 512
./CUDA_axpy5 1024

 *** function = Float AXPY (CUDA Prefetch + Page Creation + Mem Advise)
numElements = 16777216
numBlocks = 65536, numThreads = 256
Kernel time over 30 runs: avg = 0.768346 ms, min = 0.766688 ms, max = 0.770048 ms
Error count(CUDA program): 0
First 5 of Z: 0.000000 1.250000 2.500000 3.750000 5.000000 
Last 5 of Z:  267.750000 269.000000 270.250000 271.500000 272.750000 

 *** function = Float AXPY (CUDA Prefetch + Page Creation + Mem Advise)
numElements = 16777216
numBlocks = 32768, numThreads = 512
Kernel time over 30 runs: avg = 0.782662 ms, min = 0.776192 ms, max = 0.789248 ms
Error count(CUDA program): 0
First 5 of Z: 0.000000 1.250000 2.500000 3.750000 5.000000 
Last 5 of Z:  267.750000 269.000000 270.250000 271.500000 272.750000 

 *** function = Float AXPY (CUDA Prefetch + Page Creation + Mem Advise)
numElements = 16777216
numBlocks = 16384, numThreads = 1024
Kernel time over 30 runs: avg = 0.780159 ms, min = 0.778240 ms, max = 0.783872 ms
Error count(CUDA program): 0
First 5 of 

In [ ]:
%%shell
nsys profile --trace=cuda,osrt --output=/content/report_v5 --force-overwrite=true --stats=false ./CUDA_axpy5 1024
nsys stats --force-export=true --report cuda_gpu_sum,cuda_api_sum,cuda_gpu_mem_size_sum /content/report_v5.nsys-rep

 *** function = Float AXPY (CUDA Prefetch + Page Creation + Mem Advise)
numElements = 16777216
numBlocks = 16384, numThreads = 1024
Kernel time over 30 runs: avg = 0.811727 ms, min = 0.805888 ms, max = 0.826304 ms
Error count(CUDA program): 0
First 5 of Z: 0.000000 1.250000 2.500000 3.750000 5.000000 
Last 5 of Z:  267.750000 269.000000 270.250000 271.500000 272.750000 

Generating '/tmp/nsys-report-a9cc.qdstrm'
[1/1] [========================100%] report_v5.nsys-rep
Generated:
	/content/report_v5.nsys-rep
Generating SQLite file /content/report_v5.sqlite from /content/report_v5.nsys-rep
Processing [/content/report_v5.sqlite] with [/opt/nvidia/nsight-compute/2025.3.1/host/target-linux-x64/reports/cuda_gpu_sum.py]... 

 ** CUDA GPU Summary (Kernels/MemOps) (cuda_gpu_sum):

 Time (%)  Total Time (ns)  Instances  Avg (ns)   Med (ns)   Min (ns)  Max (ns)  StdDev (ns)   Category                                Operation                            
 --------  ---------------  ---------  ------